In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ln -sf /content/drive/MyDrive/colbert_ru /content/colbert_ru_drive

In [3]:
!pip install -r ./colbert_ru_drive/code_hard_negatives/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.9 MB/s eta 0:00:00
  Attempting uninstall: python-box
    Found existing installation: python-box 7.4.1
    Uninstalling python-box-7.4.1:
      Successfully uninstalled python-box-7.4.1


In [4]:
import os, sys, logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
sys.path.insert(0, "/content/colbert_ru_drive/code_hard_negatives/")

In [5]:
import json
from data import (
    _load_mmarco_examples, _load_miracl_examples,
    enrich_with_hard_negatives, RetrievalExample,
)
from config import DataConfig

DRIVE_DATA = "/content"
data_cfg = DataConfig(
    hard_negative_source="bm25",
    num_hard_negatives=7,
    cache_dir=f"{DRIVE_DATA}/cache",
    mmarco_max_triples=1_000_000
)

ru_examples = _load_mmarco_examples(data_cfg, split="train", lang="russian")
miracl_examples = _load_miracl_examples(data_cfg, split="train")
all_examples = ru_examples + miracl_examples

data/google/collections/russian_collecti(…):   0%|          | 0.00/5.77G [00:00<?, ?B/s]

data/google/queries/train/russian_querie(…):   0%|          | 0.00/62.4M [00:00<?, ?B/s]

data/triples.train.ids.small.tsv:   0%|          | 0.00/905M [00:00<?, ?B/s]

topics.miracl-v1.0-ru-train.tsv: 0.00B [00:00, ?B/s]

qrels.miracl-v1.0-ru-train.tsv: 0.00B [00:00, ?B/s]

...

miracl-corpus-v1.0-ru/docs-15.jsonl.gz:   0%|          | 0.00/79.4M [00:00<?, ?B/s]

miracl-corpus-v1.0-ru/docs-16.jsonl.gz:   0%|          | 0.00/81.4M [00:00<?, ?B/s]

miracl-corpus-v1.0-ru/docs-17.jsonl.gz:   0%|          | 0.00/82.0M [00:00<?, ?B/s]

miracl-corpus-v1.0-ru/docs-18.jsonl.gz:   0%|          | 0.00/81.0M [00:00<?, ?B/s]

miracl-corpus-v1.0-ru/docs-19.jsonl.gz:   0%|          | 0.00/6.75M [00:00<?, ?B/s]

In [6]:
corpus_texts = list({ex.positive for ex in all_examples})
print(f"Mining hard negatives over {len(corpus_texts)} docs...")
print(f"Total examples: {len(all_examples)}")

Mining hard negatives over 365815 docs...
Total examples: 1004683


In [ ]:
all_examples = enrich_with_hard_negatives(all_examples, corpus_texts, data_cfg)

BM25 scoring: 100%|██████████| 1385/1385 [1:51:51<00:00,  4.85s/it]


In [8]:
hn_path = "/content/colbert_ru_drive/datasets/hard_negatives.json"
hn_data = [
    {"query": ex.query, "positive": ex.positive, "negatives": ex.negatives, "lang": ex.lang}
    for ex in all_examples
]
with open(hn_path, "w") as f:
    json.dump(hn_data, f, ensure_ascii=False)
print(f"Hard negatives -> {hn_path} ({len(all_examples)} examples)")

Hard negatives -> /content/colbert_ru_drive/datasets/hard_negatives.json (1004683 examples)
